# 🌆 City Conditions ETL Pipeline
### A daily automated pipeline for weather + air quality analytics — Toronto, Canada

This notebook documents the full end-to-end ETL pipeline for the [City Conditions ETL Pipeline](https://github.com/Data-Netrunner/City-Conditions-ETL-Pipeline) project.

The pipeline runs automatically every day via GitHub Actions. It:
1. **Extracts** hourly weather and air quality data from the Open-Meteo API (no API key required)
2. **Transforms** the raw data — cleaning, deduplicating, and validating
3. **Loads** it into a DuckDB analytics warehouse
4. **Reports** daily KPI summaries and auto-updates charts + README

> **Note:** This notebook is a readable walkthrough of the pipeline code. The live pipeline runs automatically on GitHub — you do not need to run this notebook to operate the project. If you'd like to adapt this pipeline for your own use, feel free to fork the repository.

---


## 📁 Project Structure

```
City-Conditions-ETL-Pipeline/
├── etl/
│   ├── config.py                  # City coordinates and file paths
│   ├── extract_weather.py         # Pull hourly weather from Open-Meteo
│   ├── extract_openaq.py          # Pull hourly air quality from Open-Meteo AQ
│   ├── transform_weather.py       # Clean and validate weather data
│   ├── transform_air_quality.py   # Clean and validate air quality data
│   ├── load_weather_duckdb.py     # Upsert weather into DuckDB warehouse
│   ├── load_air_quality_duckdb.py # Upsert air quality into DuckDB warehouse
│   ├── make_charts_combined.py    # Generate 30-day time-series charts
│   ├── update_readme_weather.py   # Auto-update README with latest KPIs
│   ├── data_quality.py            # Column validation + run logging
│   └── run_weather_pipeline.py    # Main entry point — runs the full pipeline
├── sql/
│   ├── schema.sql                 # Warehouse table definitions
│   ├── kpis_combined.sql          # Daily KPI query (weather + air quality)
│   └── kpis_weather.sql           # Weather-only KPI query (reference)
├── .github/workflows/
│   └── daily_weather_etl.yml      # GitHub Actions cron schedule
├── reports/
│   ├── latest_kpis.csv            # Most recent daily KPI snapshot
│   ├── run_log.csv                # Log of every pipeline run
│   └── charts/                    # Auto-generated PNG charts
├── warehouse/
│   └── city_conditions.duckdb     # DuckDB analytics warehouse
├── data/raw/
│   └── weather_hourly.csv         # Raw weather data (not versioned)
└── requirements.txt
```

---


## 📦 Dependencies

The pipeline uses four Python libraries, all installable via pip:


In [ ]:
# Install dependencies
!pip install duckdb pandas requests matplotlib


---
## 1️⃣ Configuration — `etl/config.py`

All location settings and file paths live in one place. To adapt this pipeline for a different city, only this file needs to change.


In [ ]:
# etl/config.py

CITY     = "Toronto"
LAT      = 43.6532
LON      = -79.3832
TIMEZONE = "America/Toronto"

RAW_WEATHER_CSV = "data/raw/weather_hourly.csv"


---
## 2️⃣ Extract — Weather Data — `etl/extract_weather.py`

Pulls the past 7 days of hourly weather data from the [Open-Meteo Forecast API](https://open-meteo.com/). No API key is required. Returns temperature, precipitation, and wind speed.


In [ ]:
# etl/extract_weather.py

import requests
import pandas as pd


def fetch_weather_hourly(lat: float, lon: float, timezone: str, past_days: int = 7) -> pd.DataFrame:
    """
    Pull past N days of hourly weather from Open-Meteo.
    Returns: ts, temperature_c, precipitation_mm, windspeed_kmh
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude":  lat,
        "longitude": lon,
        "hourly":    "temperature_2m,precipitation,windspeed_10m",
        "past_days": past_days,
        "timezone":  timezone,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()

    hourly = j["hourly"]
    df = pd.DataFrame({
        "ts":               pd.to_datetime(hourly["time"]),
        "temperature_c":    hourly["temperature_2m"],
        "precipitation_mm": hourly["precipitation"],
        "windspeed_kmh":    hourly["windspeed_10m"],
    })

    return df


---
## 3️⃣ Extract — Air Quality Data — `etl/extract_openaq.py`

Pulls the past 7 days of hourly air quality data from the [Open-Meteo Air Quality API](https://open-meteo.com/en/docs/air-quality-api). Captures PM2.5, PM10, nitrogen dioxide, and ozone — all key indicators of air quality.


In [ ]:
# etl/extract_openaq.py

import requests
import pandas as pd


def fetch_air_quality_hourly(lat: float, lon: float, timezone: str, past_days: int = 7) -> pd.DataFrame:
    """
    Pull past N days of hourly air quality from Open-Meteo Air Quality API.
    Returns: ts, pm25, pm10, no2, o3
    """
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude":  lat,
        "longitude": lon,
        "hourly":    "pm2_5,pm10,nitrogen_dioxide,ozone",
        "past_days": past_days,
        "timezone":  timezone,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()

    h = j["hourly"]
    df = pd.DataFrame({
        "ts":   pd.to_datetime(h["time"]),
        "pm25": h.get("pm2_5"),
        "pm10": h.get("pm10"),
        "no2":  h.get("nitrogen_dioxide"),
        "o3":   h.get("ozone"),
    })

    return df


---
## 4️⃣ Transform — Weather — `etl/transform_weather.py`

Cleans the raw weather data: parses timestamps, removes physically impossible values (e.g. temperature outside −60°C to 60°C), deduplicates, and sorts chronologically.


In [ ]:
# etl/transform_weather.py

import pandas as pd


def clean_weather(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["ts"] = pd.to_datetime(out["ts"], errors="coerce")
    out = out.dropna(subset=["ts"])

    # Sanity bounds
    out.loc[(out["temperature_c"] < -60) | (out["temperature_c"] > 60), "temperature_c"] = None
    out.loc[out["precipitation_mm"] < 0, "precipitation_mm"] = None
    out.loc[out["windspeed_kmh"]    < 0, "windspeed_kmh"]    = None

    out = out.drop_duplicates(subset=["ts"])
    out = out.sort_values("ts").reset_index(drop=True)

    return out


---
## 5️⃣ Transform — Air Quality — `etl/transform_air_quality.py`

Cleans the raw air quality data: parses timestamps, removes negative readings (physically impossible for particulate / gas concentrations), deduplicates, and sorts.


In [ ]:
# etl/transform_air_quality.py

import pandas as pd


def clean_air_quality(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["ts"] = pd.to_datetime(out["ts"], errors="coerce")
    out = out.dropna(subset=["ts"])

    # Remove physically impossible negatives
    for col in ["pm25", "pm10", "no2", "o3"]:
        if col in out.columns:
            out.loc[out[col] < 0, col] = None

    out = out.drop_duplicates(subset=["ts"])
    out = out.sort_values("ts").reset_index(drop=True)

    return out


---
## 6️⃣ Warehouse Schema — `sql/schema.sql`

The warehouse uses three tables. `CREATE TABLE IF NOT EXISTS` makes the schema safe to re-run on every pipeline start without wiping existing data.

- **`dim_location`** — stores the city reference (Toronto = location_id 1)
- **`fact_weather_hourly`** — one row per hour of weather data
- **`fact_air_quality_hourly`** — one row per hour of air quality data

Both fact tables use a composite primary key of `(location_id, ts)`, which is what makes the upsert pattern safe — re-running the pipeline never creates duplicate rows.


In [ ]:
# sql/schema.sql

schema_sql = """
CREATE TABLE IF NOT EXISTS dim_location (
  location_id INTEGER PRIMARY KEY,
  city        VARCHAR,
  lat         DOUBLE,
  lon         DOUBLE,
  timezone    VARCHAR
);

CREATE TABLE IF NOT EXISTS fact_weather_hourly (
  location_id      INTEGER,
  ts               TIMESTAMP,
  temperature_c    DOUBLE,
  precipitation_mm DOUBLE,
  windspeed_kmh    DOUBLE,
  PRIMARY KEY (location_id, ts)
);

CREATE TABLE IF NOT EXISTS fact_air_quality_hourly (
  location_id INTEGER,
  ts          TIMESTAMP,
  pm25        DOUBLE,
  pm10        DOUBLE,
  no2         DOUBLE,
  o3          DOUBLE,
  PRIMARY KEY (location_id, ts)
);
"""


---
## 7️⃣ Load — DuckDB Warehouse — `etl/load_weather_duckdb.py` & `etl/load_air_quality_duckdb.py`

Data is loaded into DuckDB using an **upsert** pattern (`INSERT ... ON CONFLICT DO UPDATE`). This means the pipeline is fully idempotent — running it twice on the same day updates existing rows rather than creating duplicates.

[DuckDB](https://duckdb.org/) is a fast, serverless, in-process analytical database. It supports full SQL and runs entirely from a single `.duckdb` file — no server setup required.


In [ ]:
# etl/load_weather_duckdb.py

import duckdb
import pandas as pd


def init_db(db_path: str, schema_sql_path: str) -> None:
    """Create tables if they don't exist."""
    con = duckdb.connect(db_path)
    with open(schema_sql_path, "r", encoding="utf-8") as f:
        con.execute(f.read())
    con.close()


def upsert_location(db_path: str, location_id: int, city: str, lat: float, lon: float, timezone: str) -> None:
    con = duckdb.connect(db_path)
    con.execute("""
        INSERT INTO dim_location(location_id, city, lat, lon, timezone)
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT(location_id) DO UPDATE SET
          city     = excluded.city,
          lat      = excluded.lat,
          lon      = excluded.lon,
          timezone = excluded.timezone
    """, [location_id, city, lat, lon, timezone])
    con.close()


def upsert_weather(db_path: str, df_weather: pd.DataFrame, location_id: int = 1) -> None:
    con = duckdb.connect(db_path)
    df = df_weather.copy()
    df["location_id"] = location_id
    con.register("w", df)
    con.execute("""
        INSERT INTO fact_weather_hourly(location_id, ts, temperature_c, precipitation_mm, windspeed_kmh)
        SELECT location_id, ts, temperature_c, precipitation_mm, windspeed_kmh FROM w
        ON CONFLICT(location_id, ts) DO UPDATE SET
          temperature_c    = excluded.temperature_c,
          precipitation_mm = excluded.precipitation_mm,
          windspeed_kmh    = excluded.windspeed_kmh
    """)
    con.close()


In [ ]:
# etl/load_air_quality_duckdb.py

import duckdb
import pandas as pd


def upsert_air_quality(db_path: str, df_air: pd.DataFrame, location_id: int = 1) -> None:
    con = duckdb.connect(db_path)
    df = df_air.copy()
    df["location_id"] = location_id
    con.register("a", df)
    con.execute("""
        INSERT INTO fact_air_quality_hourly(location_id, ts, pm25, pm10, no2, o3)
        SELECT location_id, ts, pm25, pm10, no2, o3 FROM a
        ON CONFLICT(location_id, ts) DO UPDATE SET
          pm25 = excluded.pm25,
          pm10 = excluded.pm10,
          no2  = excluded.no2,
          o3   = excluded.o3
    """)
    con.close()


---
## 8️⃣ KPI Query — `sql/kpis_combined.sql`

After loading, a SQL query aggregates the hourly warehouse data into daily KPIs. Weather and air quality are joined by day, giving a single combined summary row per day. The query returns the last 30 days, ordered most-recent first.


In [ ]:
# sql/kpis_combined.sql

kpis_combined_sql = """
WITH daily_weather AS (
  SELECT
    DATE(ts)              AS day,
    location_id,
    AVG(temperature_c)    AS avg_temp_c,
    MAX(temperature_c)    AS max_temp_c,
    SUM(precipitation_mm) AS total_precip_mm,
    AVG(windspeed_kmh)    AS avg_windspeed_kmh,
    MAX(windspeed_kmh)    AS max_windspeed_kmh
  FROM fact_weather_hourly
  GROUP BY 1, 2
),
daily_aq AS (
  SELECT
    DATE(ts)   AS day,
    location_id,
    AVG(pm25)  AS pm25_avg,
    MAX(pm25)  AS pm25_peak,
    AVG(pm10)  AS pm10_avg,
    MAX(pm10)  AS pm10_peak,
    AVG(no2)   AS no2_avg,
    MAX(no2)   AS no2_peak,
    AVG(o3)    AS o3_avg,
    MAX(o3)    AS o3_peak
  FROM fact_air_quality_hourly
  GROUP BY 1, 2
)
SELECT
  w.day,
  w.location_id,
  w.avg_temp_c,
  w.max_temp_c,
  w.total_precip_mm,
  w.avg_windspeed_kmh,
  w.max_windspeed_kmh,
  a.pm25_avg,
  a.pm25_peak,
  a.pm10_avg,
  a.pm10_peak,
  a.no2_avg,
  a.no2_peak,
  a.o3_avg,
  a.o3_peak
FROM daily_weather w
LEFT JOIN daily_aq a
  ON  w.day         = a.day
  AND w.location_id = a.location_id
ORDER BY w.day DESC
LIMIT 30;
"""


---
## 9️⃣ Reporting — Charts — `etl/make_charts_combined.py`

Three 30-day time-series charts are generated from the KPI CSV and saved as PNG files. `matplotlib.use("Agg")` is set explicitly so the chart generation works in headless environments like GitHub Actions, which have no display.


In [ ]:
# etl/make_charts_combined.py

import os

import matplotlib
matplotlib.use("Agg")  # Required for headless environments (GitHub Actions has no display)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd


def _style_time_axis() -> None:
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.xticks(rotation=45, ha="right")
    ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.5)


def make_charts(kpis_csv: str, charts_dir: str = "reports/charts") -> None:
    os.makedirs(charts_dir, exist_ok=True)

    df = pd.read_csv(kpis_csv)
    if df.empty:
        print("No KPI data — charts skipped.")
        return

    df["day"] = pd.to_datetime(df["day"])
    df = df.sort_values("day")

    # 1) Average Temperature (30d)
    plt.figure()
    plt.plot(df["day"], df["avg_temp_c"], label="Avg Temp")
    plt.title("Average Temperature (Last 30 Days)")
    plt.xlabel("Date")
    plt.ylabel("Temperature (°C)")
    _style_time_axis()
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_dir, "avg_temp_30d.png"))
    plt.close()

    # 2) Total Precipitation (30d)
    plt.figure()
    plt.plot(df["day"], df["total_precip_mm"], label="Total Precip")
    plt.title("Total Precipitation per Day (Last 30 Days)")
    plt.xlabel("Date")
    plt.ylabel("Precipitation (mm)")
    _style_time_axis()
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_dir, "precip_30d.png"))
    plt.close()

    # 3) PM2.5 Average (30d)
    if "pm25_avg" in df.columns:
        plt.figure()
        plt.plot(df["day"], df["pm25_avg"], label="PM2.5 Avg")
        plt.title("PM2.5 Average (Last 30 Days)")
        plt.xlabel("Date")
        plt.ylabel("PM2.5 (µg/m³)")
        _style_time_axis()
        plt.legend(loc="best")
        plt.tight_layout()
        plt.savefig(os.path.join(charts_dir, "pm25_avg_30d.png"))
        plt.close()


---
## 🔟 Reporting — README Update — `etl/update_readme_weather.py`

After each run, the README is automatically rewritten with the latest KPI snapshot and updated chart images. The `_fmt()` helper handles missing values gracefully, displaying `NA` rather than crashing when air quality data isn't available for recent days.


In [ ]:
# etl/update_readme_weather.py

import math
import pandas as pd


def _fmt(x) -> str:
    """Format a numeric value for README display; returns 'NA' for missing."""
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return "NA"
        return str(round(float(x), 2))
    except Exception:
        return "NA"


def update_readme(readme_path: str, kpis_csv_path: str) -> None:
    df = pd.read_csv(kpis_csv_path)
    latest = df.iloc[0].to_dict() if not df.empty else {}

    lines = []
    lines.append("# City Conditions ETL (Daily)\n\n")
    lines.append(
        "Automated end-to-end ETL pipeline that pulls daily-updating "
        "**weather + air quality** data, loads it into a DuckDB analytics "
        "warehouse, and publishes KPI reports + charts.\n\n"
    )

    if latest:
        lines.append("## Latest KPI snapshot\n\n")
        lines.append(f"- Date: **{latest.get('day')}**\n")
        lines.append(f"- Avg Temp (°C): **{_fmt(latest.get('avg_temp_c'))}**\n")
        lines.append(f"- Max Temp (°C): **{_fmt(latest.get('max_temp_c'))}**\n")
        lines.append(f"- Total Precip (mm): **{_fmt(latest.get('total_precip_mm'))}**\n")
        lines.append(f"- Avg Wind (km/h): **{_fmt(latest.get('avg_windspeed_kmh'))}**\n")
        lines.append(f"- Max Wind (km/h): **{_fmt(latest.get('max_windspeed_kmh'))}**\n")
        if "pm25_avg" in latest:
            lines.append(f"- PM2.5 Avg (µg/m³): **{_fmt(latest.get('pm25_avg'))}**\n")
            lines.append(f"- PM2.5 Peak (µg/m³): **{_fmt(latest.get('pm25_peak'))}**\n")
        lines.append("\n")

    lines.append("## Charts (auto-updated)\n\n")
    lines.append("### Weather\n\n")
    lines.append("![Average Temperature (30d)](reports/charts/avg_temp_30d.png)\n\n")
    lines.append("![Daily Precipitation (30d)](reports/charts/precip_30d.png)\n\n")
    lines.append("### Air Quality\n\n")
    lines.append("![PM2.5 Average (30d)](reports/charts/pm25_avg_30d.png)\n\n")

    lines.append("## Outputs\n\n")
    lines.append("- `warehouse/city_conditions.duckdb`\n")
    lines.append("- `reports/latest_kpis.csv`\n")
    lines.append("- `reports/charts/*.png`\n")

    with open(readme_path, "w", encoding="utf-8") as f:
        f.writelines(lines)


---
## 1️⃣1️⃣ Data Quality & Run Logging — `etl/data_quality.py`

Two validation helpers catch problems early — before bad data reaches the warehouse. Every run (success or failure) is also appended to `reports/run_log.csv`, giving a full history of pipeline health over time.


In [ ]:
# etl/data_quality.py

import os
from datetime import datetime, timezone

import pandas as pd

LOG_PATH = "reports/run_log.csv"


def assert_required_columns(df: pd.DataFrame, required: list[str], df_name: str) -> None:
    """Raise if any expected columns are missing from the dataframe."""
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} missing required columns: {missing}")


def assert_not_empty(df: pd.DataFrame, df_name: str) -> None:
    """Raise if the dataframe is None or has no rows."""
    if df is None or df.empty:
        raise ValueError(f"{df_name} is empty — nothing to load")


def append_run_log(
    weather_rows: int,
    aq_rows: int,
    kpi_rows: int,
    status: str,
    message: str = "",
) -> None:
    """Append one row to the run log CSV after each pipeline execution."""
    os.makedirs("reports", exist_ok=True)

    row = pd.DataFrame([{
        "run_ts_utc":   datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"),
        "weather_rows": weather_rows,
        "aq_rows":      aq_rows,
        "kpi_rows":     kpi_rows,
        "status":       status,
        "message":      message,
    }])

    if os.path.exists(LOG_PATH):
        existing = pd.read_csv(LOG_PATH)
        out = pd.concat([existing, row], ignore_index=True)
    else:
        out = row

    out.to_csv(LOG_PATH, index=False)


---
## 1️⃣2️⃣ Main Pipeline Entry Point — `etl/run_weather_pipeline.py`

This is the file that GitHub Actions calls every day. It orchestrates all the steps above in sequence, wrapped in a `try/except` so that any failure is logged to `run_log.csv` before the error is re-raised (which causes the GitHub Actions run to show as failed).


In [ ]:
# etl/run_weather_pipeline.py

import os

import matplotlib
matplotlib.use("Agg")  # Must be set before any other matplotlib import

import duckdb

from etl.config import CITY, LAT, LON, TIMEZONE, RAW_WEATHER_CSV
from etl.extract_weather import fetch_weather_hourly
from etl.transform_weather import clean_weather
from etl.load_weather_duckdb import init_db, upsert_location, upsert_weather
from etl.extract_openaq import fetch_air_quality_hourly
from etl.transform_air_quality import clean_air_quality
from etl.load_air_quality_duckdb import upsert_air_quality
from etl.make_charts_combined import make_charts
from etl.update_readme_weather import update_readme
from etl.data_quality import assert_required_columns, assert_not_empty, append_run_log

DB_PATH    = "warehouse/city_conditions.duckdb"
SCHEMA_SQL = "sql/schema.sql"
KPI_SQL    = "sql/kpis_combined.sql"
KPI_OUT    = "reports/latest_kpis.csv"


def main() -> None:
    weather_rows = 0
    aq_rows      = 0
    kpi_rows     = 0

    try:
        os.makedirs("data/raw",       exist_ok=True)
        os.makedirs("warehouse",      exist_ok=True)
        os.makedirs("reports",        exist_ok=True)
        os.makedirs("reports/charts", exist_ok=True)

        # 0) Schema + location seed
        init_db(DB_PATH, SCHEMA_SQL)
        upsert_location(DB_PATH, 1, CITY, LAT, LON, TIMEZONE)

        # 1) Weather: Extract → Transform → Load
        df_weather_raw = fetch_weather_hourly(LAT, LON, TIMEZONE, past_days=7)
        assert_required_columns(
            df_weather_raw,
            ["ts", "temperature_c", "precipitation_mm", "windspeed_kmh"],
            "weather_raw",
        )
        df_weather_raw.to_csv(RAW_WEATHER_CSV, index=False)
        df_weather = clean_weather(df_weather_raw)
        assert_not_empty(df_weather, "weather_clean")
        weather_rows = len(df_weather)
        upsert_weather(DB_PATH, df_weather, location_id=1)

        # 2) Air Quality: Extract → Transform → Load
        df_aq_raw = fetch_air_quality_hourly(LAT, LON, TIMEZONE, past_days=7)
        assert_required_columns(df_aq_raw, ["ts", "pm25", "pm10", "no2", "o3"], "aq_raw")
        df_aq = clean_air_quality(df_aq_raw)
        assert_not_empty(df_aq, "aq_clean")
        aq_rows = len(df_aq)
        upsert_air_quality(DB_PATH, df_aq, location_id=1)

        # 3) Combined KPIs
        con = duckdb.connect(DB_PATH)
        with open(KPI_SQL, "r", encoding="utf-8") as f:
            sql = f.read()
        df_kpis = con.execute(sql).df()
        con.close()
        assert_not_empty(df_kpis, "kpi_output")
        kpi_rows = len(df_kpis)
        df_kpis.to_csv(KPI_OUT, index=False)

        # 4) Charts + README
        make_charts(KPI_OUT, "reports/charts")
        update_readme("README.md", KPI_OUT)

        append_run_log(weather_rows, aq_rows, kpi_rows, status="success")
        print("Pipeline complete (Weather + Air Quality + Charts).")

    except Exception as e:
        append_run_log(weather_rows, aq_rows, kpi_rows, status="failed", message=str(e))
        raise


if __name__ == "__main__":
    main()


---
## 1️⃣3️⃣ Automation — GitHub Actions — `.github/workflows/daily_weather_etl.yml`

The workflow runs automatically at **11:10 UTC every day** (~7:10 AM Toronto time). It can also be triggered manually from the GitHub Actions tab using `workflow_dispatch`.

Key design decisions:
- `concurrency` prevents two runs overlapping if one is slow
- `fetch-depth: 0` ensures git has full history for the rebase step
- `git pull --rebase --autostash` before committing prevents push conflicts
- `git diff --cached --quiet || git commit` means if there are no changes, it skips the commit cleanly


In [ ]:
# .github/workflows/daily_weather_etl.yml

workflow_yaml = """
name: Daily Weather ETL

on:
  schedule:
    - cron: "10 11 * * *"   # 11:10 UTC daily (~7:10 AM Toronto)
  workflow_dispatch:

permissions:
  contents: write

concurrency:
  group: daily-weather-etl
  cancel-in-progress: false

jobs:
  run-etl:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4
        with:
          fetch-depth: 0

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run pipeline
        env:
          PYTHONPATH: ${{ github.workspace }}
        run: |
          python etl/run_weather_pipeline.py

      - name: Commit and push updates
        run: |
          git config user.name  "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git pull --rebase --autostash origin main
          git add README.md reports/charts reports/latest_kpis.csv reports/run_log.csv
          git diff --cached --quiet || git commit -m "Daily update ETL outputs"
          git push origin main
"""


---
## ✅ Pipeline Summary

| Stage | File | What it does |
|---|---|---|
| Config | `etl/config.py` | City coordinates and file paths |
| Extract | `etl/extract_weather.py` | Pulls hourly weather from Open-Meteo |
| Extract | `etl/extract_openaq.py` | Pulls hourly air quality from Open-Meteo AQ |
| Transform | `etl/transform_weather.py` | Cleans and validates weather data |
| Transform | `etl/transform_air_quality.py` | Cleans and validates air quality data |
| Load | `etl/load_weather_duckdb.py` | Upserts weather into DuckDB |
| Load | `etl/load_air_quality_duckdb.py` | Upserts air quality into DuckDB |
| Report | `etl/make_charts_combined.py` | Generates 30-day PNG charts |
| Report | `etl/update_readme_weather.py` | Rewrites README with latest KPIs |
| Orchestrate | `etl/run_weather_pipeline.py` | Runs all stages in sequence |
| Validate | `etl/data_quality.py` | Column checks + run logging |
| Automate | `.github/workflows/daily_weather_etl.yml` | Daily cron via GitHub Actions |

---
*Built by Andre Felix · [GitHub Repository](https://github.com/Data-Netrunner/City-Conditions-ETL-Pipeline)*
